# DIY Copilot
## Lesson: Basic RAG

This lesson is intended to show you the basics of a Retrieval Augmented Generation (RAG) system. With RAG, a coding assistant finds the relevant parts of your internal knowledge – business rules, engineering guidelines, security requirements – before it answers or writes code. This is how you teach an assistant the things that are *not* in the code.

During this lesson you will learn how to ...

- implement the different building blocks of RAG
- create an ingestion pipeline for a knowledge base from the building blocks
- create a retrieval pipeline from the building blocks
- use a RAG system to answer questions and to generate code that follows your company's rules

### Set up the environment

In [ ]:
# >>> SETUP <<<
# In Google Colab, this cell
#   1. clones the workshop repository,
#   2. installs its dependencies into the Python environment of this notebook (with uv),
#   3. makes the shared code (lib) importable.
# Locally (after `uv sync` in the repository), only step 3 is needed.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/openknowledge/diy-copilot-bootcamp"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/diy-copilot")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, str(REPO_DIR / "setup" / "install_dependencies.py")], check=True)
else:
    # The repository root is the first parent directory containing the shared code
    REPO_DIR = next(p for p in Path.cwd().resolve().parents if (p / "lib").is_dir() and (p / "pyproject.toml").exists())

# The repository is not an installed package, so we add it to the module search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

In [ ]:
import uuid
from pathlib import Path

from chromadb import EphemeralClient
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from litellm import embedding

from lib.completion import generate_completion
from lib.constants import CODEBASE_PATH, KNOWLEDGE_BASE_PATH
from lib.environment import load_environment
from lib.rag import list_knowledge_base_files
from lib.retry import call_with_retry

In [ ]:
# Models
default_models = load_environment()
generation_model = default_models["generation_model"]
embedding_model = default_models["embedding_model"]
print(f"Generation model: {generation_model}")
print(f"Embedding model:  {embedding_model}")

# Generation parameters
TEMPERATURE = 0.7
MAX_OUTPUT_TOKENS = 1000

# Retrieval parameters
TOP_K = 3
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# Prompts
SYSTEM_PROMPT = "You are a helpful coding assistant for the developers of our company."

## Generate a response without internal knowledge

In [ ]:
# Let's ask a specific question about the rules of our company.
user_prompt = "At our company, what happens if nobody checks in to a meeting room booking?"
# The answer should mention that the booking is released 10 minutes after the start.

print(generate_completion(user_prompt=user_prompt, system_prompt=SYSTEM_PROMPT, model_name=generation_model))

## Define helpers

In [ ]:
# This will be the chromadb collection we use as a knowledge base. The EphemeralClient keeps everything in memory.
chromadb_collection = EphemeralClient().get_or_create_collection(name="default")


def peek_knowledgebase(n: int = 3):
    """Shows the first n items of the knowledge base."""
    items = chromadb_collection.peek(limit=n)
    for document, metadata, embeddings in zip(items["documents"], items["metadatas"], items["embeddings"]):
        print(f"----- {metadata} -----\n{document}\n{embeddings}\n")

### Define RAG Building Blocks

In [ ]:
# Building Block "Loading": Get the content of a document of the knowledge base.
def load_file_content(file_name: str) -> str:
    """Loads the content of a document of the knowledge base.

    Parameters
    ----------
    file_name : str
        The path of the document relative to the root of the knowledge base
    Returns
    -------
    file_content : str
        The content of the file as a string
    """
    return (KNOWLEDGE_BASE_PATH / file_name).read_text(encoding="utf-8")

#### Chunking structured documents
A plain text splitter cuts wherever the chunk is full – possibly in the middle of a section. LangChain's `RecursiveCharacterTextSplitter.from_language()` knows the syntax of Markdown (and of many programming languages) and prefers to split at headings, so that a chunk tends to contain a complete section. Let's compare both.

In [ ]:
# Building Block "Chunking": Split the content into smaller chunks
def do_chunk(text: str, file_name: str) -> list[str]:
    """Chunks a text with a splitter that matches the type of the file.

    Parameters
    ----------
    text : str
        The text to be chunked
    file_name : str
        The name of the file; its extension determines the splitter
    Returns
    -------
    chunks: list[str]
        The created chunks
    """
    language = {".py": Language.PYTHON, ".md": Language.MARKDOWN}.get(Path(file_name).suffix)
    if language is None:
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    else:
        text_splitter = RecursiveCharacterTextSplitter.from_language(
            language=language, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
    return text_splitter.split_text(text)

In [ ]:
# Compare a plain splitter with the Markdown-aware splitter on the booking policy
sample = load_file_content("domain/room-booking-policy.md")
plain_chunks = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP).split_text(sample)
markdown_chunks = do_chunk(sample, "domain/room-booking-policy.md")

print("First lines of each chunk (plain splitter):")
for chunk in plain_chunks:
    print("   ", chunk.splitlines()[0])
print("\nFirst lines of each chunk (Markdown splitter):")
for chunk in markdown_chunks:
    print("   ", chunk.splitlines()[0])

In [ ]:
# Building Block "Embedding": Create multi-dimensional embeddings for a given chunk.
def do_embed(chunk: str) -> list[float]:
    """Embeds a given chunk and returns the embedding.

    Parameters
    ----------
    chunk : str
        The chunk to be embedded
    Returns
    -------
    embedding: list[float]
        The created embedding
    """
    # call_with_retry waits and retries if the free tier rate limit is reached
    response = call_with_retry(embedding, model=embedding_model, input=chunk)
    return response["data"][0]["embedding"]


def do_batch_embed(chunks: list[str], batch_size: int = 100) -> list[list[float]]:
    """Embeds a list of chunks in batches and returns the embeddings.

    Sending many chunks per request is much faster and saves requests - important
    for the rate limits of the free tier.

    Parameters
    ----------
    chunks : list[str]
        The chunks to be embedded
    batch_size : int
        The number of chunks per request (max. 100 for Gemini)
    Returns
    -------
    embeddings: list[list[float]]
        The created embeddings
    """
    all_embeddings = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        response = call_with_retry(embedding, model=embedding_model, input=batch)
        all_embeddings.extend(data["embedding"] for data in response["data"])
    return all_embeddings

In [ ]:
# Building Block "Knowledge base": Store embeddings, chunks and metadata in a vector store
def persist_embeddings(chunks: list[str], embeddings: list[list[float]], metadatas: list[dict]) -> None:
    """Persists the chunks, their embeddings and metadata in the knowledge base.

    Parameters
    ----------
    chunks : list[str]
        The chunks to be persisted
    embeddings: list[list[float]]
        The corresponding embeddings to be persisted
    metadatas: list[dict]
        The corresponding metadata, e.g. the document the chunk comes from
    """
    ids = [str(uuid.uuid4()) for _ in chunks]
    chromadb_collection.add(ids=ids, documents=chunks, embeddings=embeddings, metadatas=metadatas)

In [ ]:
# Building Block "Augmentation": Create an updated prompt by merging the original user input with the provided context
def augment(user_input: str, context: list[str]) -> str:
    """Augments a given user input by merging it with the provided context.

    Parameters
    ----------
    user_input : str
        The user input to be augmented
    context : list[str]
        The context to be merged with the user input
    Returns
    -------
    augmented_prompt: str
        The created augmented prompt
    """
    prepared_context = "\n\n---\n\n".join(context)
    return f"""Answer the question as precisely as possible, using only the provided
context from our internal knowledge base. Mention the documents you used. If the answer
is not in the context, say "The answer is not available in the context." and do
not make anything up.

Context:
{prepared_context}

Question:
{user_input}

Answer:
"""

In [ ]:
# Building Block "Top-k Fetching": Get the k semantically closest chunks to the user input from the knowledge base
def do_top_k_fetching(user_input_embedding: list[float], top_k: int) -> list[str]:
    """Fetches the k semantically closest chunks to the user input from the knowledge base.

    Parameters
    ----------
    user_input_embedding : list[float]
        The embedding of the user input
    top_k : int
        The number of semantically closest chunks to be fetched

    Returns
    -------
    context: list[str]
        The fetched chunks, each prefixed with the document it comes from
    """
    # Chroma allows querying for multiple embeddings at once. We query for one,
    # so we only use the first list of results.
    result = chromadb_collection.query(query_embeddings=[user_input_embedding], n_results=top_k)
    return [
        f"Document: {metadata['source']}\n{document}"
        for document, metadata in zip(result["documents"][0], result["metadatas"][0])
    ]

In [ ]:
# Building Block "Generation": Use the generation model to create a response
def generate_response(prompt: str) -> None:
    """Generates a response for a given prompt and prints it.

    Parameters
    ----------
    prompt : str
        The prompt to be used for the generation
    """
    response = generate_completion(
        model_name=generation_model,
        system_prompt=SYSTEM_PROMPT,
        user_prompt=prompt,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )
    print(response)

### Create the ingestion pipeline

In [ ]:
def do_ingestion(file_names: list[str]) -> None:
    """Ingests a list of documents of the knowledge base.

    Parameters
    ----------
    file_names : list[str]
        The paths of the documents to be ingested, relative to the root of the knowledge base
    """
    all_chunks, all_metadatas = [], []
    for file_name in file_names:
        # Load the document content
        file_content = load_file_content(file_name)

        # Chunk the content into smaller chunks
        chunks = do_chunk(file_content, file_name)
        all_chunks.extend(chunks)
        all_metadatas.extend({"source": file_name} for _ in chunks)

    # Embed the chunks of all files together to save requests
    embeddings = do_batch_embed(all_chunks)

    # Persist the chunks, embeddings and metadata in the knowledge base
    persist_embeddings(all_chunks, embeddings, all_metadatas)
    print(f"Ingested {len(all_chunks)} chunks from {len(file_names)} documents.")

### Perform ingestion

In [ ]:
# Ingest all documents of the knowledge base: domain, engineering and security
file_names = list_knowledge_base_files(KNOWLEDGE_BASE_PATH)
do_ingestion(file_names)

In [ ]:
# Use the helper function to peek into the knowledge base
peek_knowledgebase()

### Exercise 01: Create the RAG pipeline
In this exercise you will create a RAG pipeline for retrieving relevant chunks and generating a grounded response.

In [ ]:
def do_rag(user_input: str, verbose: bool = False) -> None:
    """Runs the RAG pipeline with a given user input and prints the response.

    Parameters
    ----------
    user_input : str
        The user input to be used for the RAG pipeline
    verbose : bool, optional [default: False]
        Whether to print details of the RAG process or not
    """
    # TODO: Embed the user input
    user_input_embedding = None

    # TODO: "R" like "Retrieval": Get the k semantically closest chunks to the user input from the knowledge base
    context = None
    if verbose:
        print("Retrieved context:")
        for chunk in context:
            print(f"----------\n{chunk}")
        print("----------\n")

    # TODO: "A" like "Augmented": Create the augmented prompt
    augmented_prompt = None

    # TODO: "G" like "Generation": Generate a response
    None

In [ ]:
# Define the user input. This should be a question about our internal knowledge.
user_input = "At our company, what happens if nobody checks in to a meeting room booking?"
# The answer should mention that the booking is released 10 minutes after the start.

# TODO: Perform RAG
None

### Exercise 02: Ask your own questions
Ask the RAG system some questions of your own – about the business rules, the engineering guidelines or the security requirements. Also try a question whose answer is **not** in the knowledge base. Does the assistant admit that it doesn't know?

In [ ]:
# TODO: Ask your own questions
None